# Data Preprocessing

This file contains the preprocessing for the players visualizations

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("data/player_valuations.csv")  # Replace with your file

# Convert 'date' to datetime
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df["player_id"] = df["player_id"].astype(int)

# Remove missing dates
df = df.dropna(subset=["date"])

# Fill missing market values using forward fill per player
df["market_value_in_eur"] = df.groupby("player_id")["market_value_in_eur"].fillna(method="ffill")

# Convert market value to millions
df["market_value_in_millions"] = df["market_value_in_eur"] / 1_000_000

# Sort by player and date
df = df.sort_values(by=["player_id", "date"])

# Save processed data
df.to_csv("processed_data/player_valuations.csv", index=False)



#################################################################################
# Create a summary of the data for each player get # of red cards, yellow cards, assists, goals, and minutes played

import pandas as pd

base_path = "./data/"
appearances = pd.read_csv(base_path + "appearances.csv")
events = pd.read_csv(base_path + "game_events.csv")
players = pd.read_csv(base_path + "players.csv")

# Normalize event types
events["type_norm"] = events["type"].astype(str).str.strip().str.lower()

# 🥅 Goals
goals = events[(events["type_norm"] == "goals") & (events["player_id"].notnull())]
goal_counts = goals.groupby("player_id").size().reset_index(name="goals")

# 🎯 Assists
assists = events[events["player_assist_id"].notnull()]
assist_counts = assists.groupby("player_assist_id").size().reset_index(name="assists")
assist_counts.rename(columns={"player_assist_id": "player_id"}, inplace=True)

# ⏱️ Minutes (90 per appearance)
appearances["estimated_minutes"] = 90
minutes = appearances.groupby("player_id")["estimated_minutes"].sum().reset_index(name="minutes")

# 🟨🟥 Cards
card_counts = appearances.groupby("player_id")[["yellow_cards", "red_cards"]].sum().reset_index()

# 🧠 Merge all
summary = pd.merge(goal_counts, assist_counts, on="player_id", how="outer")
summary = pd.merge(summary, minutes, on="player_id", how="outer")
summary = pd.merge(summary, card_counts, on="player_id", how="outer")
summary = pd.merge(summary, players[["player_id", "name"]], on="player_id", how="left")

# Fill missing and convert to int
summary.fillna(0, inplace=True)
summary[["goals", "assists", "yellow_cards", "red_cards", "minutes"]] = summary[
    ["goals", "assists", "yellow_cards", "red_cards", "minutes"]
].astype(int)

# Final structure
summary = summary[["player_id", "name", "goals", "assists", "yellow_cards", "red_cards", "minutes"]]
summary.to_csv("player_summary.csv", index=False)
print("✅ Saved to player_summary.csv")

# 🏆 Print Top 15 per category
def print_top_15(column, label):
    print(f"\n🏆 Top 15 Players by {label}:\n")
    top = summary.sort_values(column, ascending=False).head(15)
    print(top[["name", column]].to_string(index=False))


In [5]:
import pandas as pd

# Step 1: Load Datasets
# Clean lines before parsing
with open("../data/players.csv", "r", encoding="utf-8") as f:
    cleaned_lines = [line.strip().strip('"') for line in f]

# Re-parse cleaned lines with pandas
from io import StringIO
cleaned_csv = StringIO("\n".join(cleaned_lines))
players = pd.read_csv(cleaned_csv, quotechar='"', doublequote=True, escapechar='\\', engine='python')


game_events = pd.read_csv("../data/game_events.csv")
appearances = pd.read_csv("../data/appearances.csv")
club_logos = pd.read_csv("../data/club_logos.csv")
competition_logos = pd.read_csv("../data/competition_logos.csv")

# Step 2: Preprocess `game_events` - Extract Goals, Assists, Cards
goal_events = game_events[game_events['type'] == 'Goals']
goals = goal_events.groupby('player_id').size().rename('goals')
assists = goal_events['player_assist_id'].value_counts().rename('assists')

cards = game_events[game_events['type'] == 'Cards']
yellow_cards = cards[cards['description'].str.contains("Yellow", na=False)].groupby('player_id').size().rename('yellow_cards')
red_cards = cards[cards['description'].str.contains("Red", na=False)].groupby('player_id').size().rename('red_cards')

# Step 3: Preprocess `appearances` - Aggregate by Player
app_stats = appearances.groupby('player_id').agg({
    'goals': 'sum',
    'assists': 'sum',
    'red_cards': 'sum',
    'yellow_cards': 'sum',
    'minutes_played': 'sum',
}).fillna(0)

# ✅ Count unique games per player (fixes overcounting)
unique_appearances = appearances.drop_duplicates(subset=['player_id', 'game_id'])
total_appearances = unique_appearances.groupby('player_id').size().rename('appearances')

# ✅ Total minutes possible = number of unique appearances × 90
total_minutes = total_appearances * 90
total_minutes.name = 'total_minutes'

# Step 4: Combine All Event Data into One Stats Table
event_stats = pd.concat([goals, assists, yellow_cards, red_cards], axis=1).fillna(0)
event_stats.index = event_stats.index.astype(int)

combined_stats = event_stats.add(app_stats, fill_value=0).fillna(0)
combined_stats = combined_stats.join(total_appearances)
combined_stats = combined_stats.join(total_minutes)
combined_stats = combined_stats.reset_index().rename(columns={'index': 'player_id'})

# Step 5: Select player metadata
player_info = players[[ 
    'player_id', 'name', 'country_of_birth', 
    'country_of_citizenship', 'date_of_birth', 'position', 'foot', 
    'height_in_cm', 'current_club_id', 'current_club_name', 'current_club_domestic_competition_id', 'market_value_in_eur', 
    'highest_market_value_in_eur', 'image_url',
]]

player_info['player_id'] = pd.to_numeric(player_info['player_id'], errors='coerce')
player_info = player_info.dropna(subset=['player_id'])
player_info['player_id'] = player_info['player_id'].astype(int)

# Step 6: Merge with club logos
club_logos['club_id'] = pd.to_numeric(club_logos['club_id'], errors='coerce')
club_logos = club_logos.dropna(subset=['club_id'])
club_logos['club_id'] = club_logos['club_id'].astype(int)

player_info = pd.merge(
    player_info,
    club_logos.rename(columns={'club_id': 'current_club_id', 'logo_url': 'club_logo_url'}),
    on='current_club_id',
    how='left'
)

# Rename column to match for merging
player_info = pd.merge(
    player_info,
    competition_logos,
    left_on='current_club_domestic_competition_id',
    right_on='competition_id',
    how='left'
)


# Step 9: Merge stats with metadata
player_summary = pd.merge(player_info, combined_stats, on='player_id', how='left')
player_summary = player_summary.fillna(0)

# Step 10: Save
player_summary.to_csv("../processed_data/player_summary.csv", index=False)

# Step 11: Preview
player_summary.head(10)


ParserError: ',' expected after '"'

In [12]:
import pandas as pd

# Step 1: Load Datasets
players = pd.read_csv("../data/players.csv")
game_events = pd.read_csv("../data/game_events.csv")
appearances = pd.read_csv("../data/appearances.csv")
club_logos = pd.read_csv("../data/club_logos.csv")  # <-- NEW

# Step 2: Preprocess `game_events` - Extract Goals, Assists, Cards
# Goals and Assists
goal_events = game_events[game_events['type'] == 'Goals']
goals = goal_events.groupby('player_id').size().rename('goals')
assists = goal_events['player_assist_id'].value_counts().rename('assists')

# Cards
cards = game_events[game_events['type'] == 'Cards']
yellow_cards = cards[cards['description'].str.contains("Yellow", na=False)].groupby('player_id').size().rename('yellow_cards')
red_cards = cards[cards['description'].str.contains("Red", na=False)].groupby('player_id').size().rename('red_cards')

# Step 3: Preprocess `appearances` - Aggregate by Player
app_stats = appearances.groupby('player_id').agg({
    'goals': 'sum',
    'assists': 'sum',
    'red_cards': 'sum',
    'yellow_cards': 'sum',
    'minutes_played': 'sum',
}).fillna(0)

# ✅ Count unique games per player (fixes overcounting)
# Assumes 'match_id' exists in appearances
unique_appearances = appearances.drop_duplicates(subset=['player_id', 'game_id'])
total_appearances = unique_appearances.groupby('player_id').size().rename('appearances')

# ✅ Total minutes possible = number of unique appearances × 90
total_minutes = total_appearances * 90
total_minutes.name = 'possible_minutes'


# Step 4: Combine All Event Data into One Stats Table
event_stats = pd.concat([goals, assists, yellow_cards, red_cards], axis=1).fillna(0)
event_stats.index = event_stats.index.astype(int)

combined_stats = event_stats.add(app_stats, fill_value=0).fillna(0)
combined_stats = combined_stats.reset_index().rename(columns={'index': 'player_id'})

# Step 5: Select player metadata
player_info = players[[ 
    'player_id', 'name', 'country_of_birth', 
    'country_of_citizenship', 'date_of_birth', 'position', 'foot', 
    'height_in_cm', 'current_club_id', 'current_club_name', 'market_value_in_eur', 
    'highest_market_value_in_eur', 'image_url',
]]

# Clean player_id
player_info['player_id'] = pd.to_numeric(player_info['player_id'], errors='coerce')
player_info = player_info.dropna(subset=['player_id'])
player_info['player_id'] = player_info['player_id'].astype(int)

# Step 6: Merge with club logos using club_id
club_logos['club_id'] = pd.to_numeric(club_logos['club_id'], errors='coerce')
club_logos = club_logos.dropna(subset=['club_id'])
club_logos['club_id'] = club_logos['club_id'].astype(int)

player_info = pd.merge(
    player_info,
    club_logos.rename(columns={'club_id': 'current_club_id', 'logo_url': 'club_logo_url'}),
    on='current_club_id',
    how='left'
)

# Step 7: Infer player's main competition from appearances
# (Optional: if `players` doesn't already have a competition_id)
main_competition = (
    appearances.groupby('player_id')['competition_id']
    .agg(lambda x: x.value_counts().idxmax())  # most frequent competition per player
    .reset_index()
)

# Step 8: Merge competition logos
competition_logos = competition_logos[['competition_id', 'competition_logo_url']]
player_info = pd.merge(player_info, main_competition, on='player_id', how='left')
player_info = pd.merge(
    player_info,
    competition_logos,
    on='competition_id',
    how='left'
)

# Step 9: Merge stats with metadata
player_summary = pd.merge(player_info, combined_stats, on='player_id', how='left')
player_summary = player_summary.fillna(0)

# Step 8: Save
player_summary.to_csv("../processed_data/player_summary.csv", index=False)

# Step 9: Preview
player_summary.head(10)


C:\Users\rayan\AppData\Local\Temp\ipykernel_6136\3218847741.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_info['player_id'] = pd.to_numeric(player_info['player_id'], errors='coerce')


,player_id,name,country_of_birth,country_of_citizenship,date_of_birth,position,foot,height_in_cm,current_club_id,current_club_name,...,image_url,club_name,club_logo_url,competition_id,competition_logo_url,assists,goals,minutes_played,red_cards,yellow_cards
0,10,Miroslav Klose,Poland,Germany,1978-06-09 00:00:00,Attack,right,184.0,398.0,Società Sportiva Lazio S.p.A.,...,https://img.a.transfermarkt.technology/portrai...,Società Sportiva Lazio S.p.A.,https://tmssl.akamaized.net//images/wappen/hea...,IT1,https://tmssl.akamaized.net//images/logo/heade...,50.0,96.0,8808.0,0.0,38.0
1,26,Roman Weidenfeller,Germany,Germany,1980-08-06 00:00:00,Goalkeeper,left,190.0,16.0,Borussia Dortmund,...,https://img.a.transfermarkt.technology/portrai...,Borussia Dortmund,https://tmssl.akamaized.net//images/wappen/hea...,L1,https://tmssl.akamaized.net//images/logo/heade...,0.0,0.0,13508.0,4.0,8.0
2,65,Dimitar Berbatov,Bulgaria,Bulgaria,1981-01-30 00:00:00,Attack,0,0.0,1091.0,Panthessalonikios Athlitikos Omilos Konstantin...,...,https://img.a.transfermarkt.technology/portrai...,Panthessalonikios Athlitikos Omilos Konstantin...,https://tmssl.akamaized.net//images/wappen/hea...,GB1,https://tmssl.akamaized.net//images/logo/heade...,27.0,76.0,8788.0,2.0,22.0
3,77,Lúcio,Brazil,Brazil,1978-05-08 00:00:00,Defender,0,0.0,506.0,Juventus Football Club,...,https://img.a.transfermarkt.technology/portrai...,Juventus Football Club,https://tmssl.akamaized.net//images/wappen/hea...,CL,https://tmssl.akamaized.net//images/logo/heade...,0.0,0.0,307.0,0.0,0.0
4,80,Tom Starke,East Germany (GDR),Germany,1981-03-18 00:00:00,Goalkeeper,right,194.0,27.0,FC Bayern München,...,https://img.a.transfermarkt.technology/portrai...,0,0,L1,https://tmssl.akamaized.net//images/logo/heade...,0.0,0.0,1080.0,0.0,0.0
5,109,Dedê,Brazil,Brazil,1978-04-18 00:00:00,Defender,0,0.0,825.0,Eskisehirspor,...,https://img.a.transfermarkt.technology/portrai...,Eskisehirspor,https://tmssl.akamaized.net//images/wappen/hea...,TR1,https://tmssl.akamaized.net//images/logo/heade...,5.0,2.0,3584.0,0.0,8.0
6,123,Christoph Metzelder,Germany,Germany,1980-11-05 00:00:00,Defender,0,0.0,33.0,FC Schalke 04,...,https://img.a.transfermarkt.technology/portrai...,FC Schalke 04,https://tmssl.akamaized.net//images/wappen/hea...,L1,https://tmssl.akamaized.net//images/logo/heade...,2.0,0.0,427.0,0.0,0.0
7,132,Tomas Rosicky,CSSR,Czech Republic,1980-10-04 00:00:00,Midfield,both,179.0,11.0,Arsenal Football Club,...,https://img.a.transfermarkt.technology/portrai...,Arsenal Football Club,https://tmssl.akamaized.net//images/wappen/hea...,GB1,https://tmssl.akamaized.net//images/logo/heade...,8.0,18.0,3987.0,0.0,26.0
8,162,Marc Ziegler,Germany,Germany,1976-06-13 00:00:00,Goalkeeper,right,193.0,79.0,Verein für Bewegungsspiele Stuttgart 1893,...,https://img.a.transfermarkt.technology/portrai...,Verein für Bewegungsspiele Stuttgart 1893,https://tmssl.akamaized.net//images/wappen/hea...,0,0,0.0,0.0,0.0,0.0,0.0
9,215,Roque Santa Cruz,Paraguay,Paraguay,1981-08-16 00:00:00,Attack,right,193.0,1084.0,Málaga CF,...,https://img.a.transfermarkt.technology/portrai...,Málaga CF,https://tmssl.akamaized.net//images/wappen/hea...,ES1,https://tmssl.akamaized.net//images/logo/heade...,16.0,52.0,6038.0,0.0,6.0


In [4]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("data/game_lineups.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower()

# Filter only starting players (who actually played)
starting_players = df[df["type"] == "starting_lineup"]

# Remove duplicate appearances per game/player/position
starting_players = starting_players.drop_duplicates(subset=["game_id", "player_id", "position"])

# Count matches per player per position
position_counts = (
    starting_players
    .groupby(["player_id", "position"])
    .agg(matches_played=("game_id", "nunique"))
    .reset_index()
)

# Pivot: one row per player, one column per position
pivoted = position_counts.pivot(index="player_id", columns="position", values="matches_played").fillna(0).astype(int)

# Add player names back (optional)
player_names = df[["player_id", "player_name"]].drop_duplicates()
pivoted = pivoted.reset_index().merge(player_names, on="player_id", how="left")

# Reorder columns: player_id, player_name, position columns
cols = ["player_id", "player_name"] + [col for col in pivoted.columns if col not in ["player_id", "player_name"]]
pivoted = pivoted[cols]

# Show the final table
pivoted.to_csv("../processed_data/position_count.csv", index=False)

In [ ]:
# Calculate the maximum values for the Hexagonal Stats on the compaire page

import pandas as pd

# Load datasets
stats = pd.read_csv("player_stats.csv")
players = pd.read_csv("players.csv")
valuations = pd.read_csv("player_valuations.csv")

# Calculate age from date_of_birth
players["date_of_birth"] = pd.to_datetime(players["date_of_birth"], errors='coerce')
today = pd.to_datetime("today")
players["age"] = players["date_of_birth"].apply(
    lambda dob: today.year - dob.year if pd.notnull(dob) else None
)

# Compute max values for radar metrics
max_goals = stats["nr_of_goals"].astype(float).max()
max_assists = stats["assists"].astype(float).max()
max_reds = stats["red_cards"].astype(float).max()
max_yellows = stats["yellow_cards"].astype(float).max()
max_aggressivity = max_reds * 2 + max_yellows
max_age = players["age"].max()
max_wealth = valuations["market_value_in_eur"].astype(float).max()
max_win_percent = 100.0  

# Output
max_values = {
    "Win%": max_win_percent,
    "Avg Age": max_age,
    "Wealth": max_wealth,
    "Aggressivity": max_aggressivity,
    "Teamwork (Assists)": max_assists,
    "Scoring (Goals)": max_goals
}

# Save to CSV
pd.DataFrame(max_values.items(), columns=["Metric", "Max Value"]).to_csv("max_metric_values.csv", index=False)
